# Constrained Cobb-Douglas with MPC — Theory and Hold-Out Results

> **Learning Objectives:**
>
> - Formulate the constrained Cobb-Douglas allocation problem (budget + covariance + turnover + concentration) and recognize it as a convex program solvable in milliseconds at K=22.
> - Read the MPC discipline (forward projection band + trigger conditions) as a discipline that converts a clock-driven rebalancer into an event-driven one.
> - Compare the 6 strategies head-to-head on the 2025-2026 hold-out and interpret what each pairwise difference isolates (constraint effect vs trigger effect).


## Section 1: Theory Recap

The constrained Cobb-Douglas allocator and MPC discipline are described in detail in `docs/superpowers/specs/2026-05-17-constrained-cobb-douglas-design.md` and the source spec `constrained_cobb_douglas.md`.


In [ ]:
include("Include.jl")


In [ ]:
# Load all artifacts
_check_artifact(joinpath(_PATH_TO_ARTIFACTS, "sim_calibration.jld2"))
_check_artifact(joinpath(_PATH_TO_ARTIFACTS, "frozen_basket.jld2"))
_check_artifact(joinpath(_PATH_TO_ARTIFACTS, "backtest_results.jld2"))

sim_calib = load_results(joinpath(_PATH_TO_ARTIFACTS, "sim_calibration.jld2"))
basket    = load_results(joinpath(_PATH_TO_ARTIFACTS, "frozen_basket.jld2"))
bt        = load_results(joinpath(_PATH_TO_ARTIFACTS, "backtest_results.jld2"))
println("Basket: ", basket["tickers"])
println("Hold-out: ", bt["config"]["hold_out_start"], " to ", bt["config"]["hold_out_end"], " (", bt["config"]["n_days"], " days)")


## Section 2: Headline Bake-Off (after-cost, after-tax)


In [ ]:
rows = NamedTuple[]
for (name, r) in bt["results"]
    sm = r.summary
    push!(rows, (Strategy = name,
        Sharpe = round(sm.ann_sharpe; digits=3),
        AnnRet_pct = round(sm.ann_return*100; digits=2),
        MaxDD_pct = round(sm.max_drawdown*100; digits=1),
        Turnover = round(sm.ann_turnover; digits=3),
        N_trig = sm.n_mpc_triggers))
end
sort!(rows; by = r -> -r.Sharpe)
pretty_table(DataFrame(rows); backend = :text)


## Section 3: Wealth Curves


In [ ]:
p = plot(legend = :outerright, size = (1080, 540),
          xlabel = "Trading day", ylabel = "Wealth (after-cost, after-tax)")
for (name, r) in bt["results"]
    plot!(p, r.wealth_after_cost_aftertax; label = name, lw = 1.4)
end
p


## Section 4: MPC Trigger Reasons


In [ ]:
for (name, r) in bt["results"]
    if !isempty(r.trigger_log)
        reasons = [t.reason for t in r.trigger_log if t.fired]
        if !isempty(reasons)
            counts = Dict(rs => count(==(rs), reasons) for rs in unique(reasons))
            println(rpad(name, 35), "  ", counts)
        end
    end
end


## Section 5: Multi-Seed Backtest Distribution

The two MPC strategies depend on `BACKTEST_RNG_SEED` through their `forward_project` Monte Carlo paths. To honestly report performance we run each strategy across 20 seeds and report the distribution of outcomes. Non-MPC strategies (EW, MinVar, UnconstrainedCD, CostAwareMV) are deterministic and report a single value.

In [ ]:
_check_artifact(joinpath(_PATH_TO_ARTIFACTS, "backtest_mc_results.jld2"))
bt_mc = load_results(joinpath(_PATH_TO_ARTIFACTS, "backtest_mc_results.jld2"))
n_seeds = bt_mc["config"]["n_seeds"]

rows = NamedTuple[]
for (name, agg) in bt_mc["summary"]
    sh = agg["sharpe_mc"]
    dd = agg["max_dd_mc"]
    wt = agg["W_T_over_W0_mc"]
    push!(rows, (Strategy = name,
        Sharpe_min = round(minimum(sh); digits=3),
        Sharpe_med = round(median(sh); digits=3),
        Sharpe_max = round(maximum(sh); digits=3),
        MaxDD_med_pct = round(median(dd)*100; digits=1),
        WT_W0_med = round(median(wt); digits=3),
        nTrig_med = round(median(agg["n_mpc_triggers_mc"]); digits=0)))
end
sort!(rows; by = r -> -r.Sharpe_med)
pretty_table(DataFrame(rows); backend = :text)

### Sharpe distribution histogram — ConstrainedCDWithMPCStrategy

In [ ]:
sh = bt_mc["summary"]["ConstrainedCDWithMPCStrategy"]["sharpe_mc"]
histogram(sh; bins = 10, xlabel = "Hold-out Sharpe", ylabel = "Seed count",
    title = "ConstrainedCDWithMPCStrategy Sharpe across $n_seeds seeds",
    legend = false, size = (900, 420))

## Disclaimer

This content is for educational purposes only and does not constitute investment advice.